# Modified Langevin Score Matching / Diffusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FloppingCode/modified-langevin-score-matching/blob/main/notebook.ipynb)

In [ ]:
import os, sys

# When running in Colab, clone the repo and add it to the path
if "google.colab" in sys.modules:
    if not os.path.exists("modified-langevin-score-matching"):
        !git clone https://github.com/FloppingCode/modified-langevin-score-matching.git
    sys.path.insert(0, "modified-langevin-score-matching")
else:
    sys.path.insert(0, ".")

import torch
from dsm import (
    make_dataset,
    make_dataloader,
    ScoreNetwork,
    GeometricNoiseSchedule,
    dsm_loss,
    train,
    annealed_langevin_dynamics,
)
from dsm.visualization import (
    plot_samples,
    plot_score_field,
    plot_training_curves,
    plot_sampling_trajectory,
)

print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Configuration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = dict(
    # Dataset
    dataset="8gaussians",  # try: "moons", "swiss_roll", "circles", "8gaussians"
    n_samples=10_000,
    data_dim=2,
    # Noise schedule
    sigma_min=0.01,
    sigma_max=1.0,
    num_noise_levels=10,
    # Model
    hidden_dim=256,
    num_res_blocks=3,
    # Training
    n_epochs=200,
    batch_size=512,
    lr=1e-3,
    # Sampling
    n_generated=2000,
    steps_per_sigma=100,
)

In [ ]:
dataset = make_dataset(CONFIG["dataset"], n_samples=CONFIG["n_samples"])
dataloader = make_dataloader(dataset, batch_size=CONFIG["batch_size"])

# Visualize
import matplotlib.pyplot as plt

data_tensor = dataset.tensors[0]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(data_tensor[:, 0], data_tensor[:, 1], s=1, alpha=0.5)
ax.set_title(f'Dataset: {CONFIG["dataset"]} ({CONFIG["n_samples"]} points)')
ax.set_aspect("equal")
plt.show()
print(f"Data shape: {data_tensor.shape}, range: [{data_tensor.min():.2f}, {data_tensor.max():.2f}]")

In [ ]:
noise_schedule = GeometricNoiseSchedule(
    sigma_min=CONFIG["sigma_min"],
    sigma_max=CONFIG["sigma_max"],
    num_levels=CONFIG["num_noise_levels"],
)

model = ScoreNetwork(
    data_dim=CONFIG["data_dim"],
    hidden_dim=CONFIG["hidden_dim"],
    num_res_blocks=CONFIG["num_res_blocks"],
)

n_params = sum(p.numel() for p in model.parameters())
print(f"ScoreNetwork: {n_params:,} parameters")
print(f"Noise levels (σ): {noise_schedule.sigmas.tolist()}")

## Train

In [ ]:
history = train(
    model,
    dataloader,
    noise_schedule,
    n_epochs=CONFIG["n_epochs"],
    lr=CONFIG["lr"],
    device=DEVICE,
    log_every=20,
)

In [ ]:
plot_training_curves(history)
plt.show()

## Score Field Visualization

Arrows should point toward data clusters at low σ and form a smooth radial field at high σ.

In [ ]:
# Plot score field at a few noise levels (largest, middle, smallest)
sigmas_to_plot = [
    noise_schedule.sigmas[0].item(),   # largest σ
    noise_schedule.sigmas[len(noise_schedule.sigmas) // 2].item(),  # middle σ
    noise_schedule.sigmas[-1].item(),  # smallest σ
]

for sigma in sigmas_to_plot:
    plot_score_field(model, sigma, device=DEVICE, data=data_tensor)
    plt.show()

## Sample via Annealed Langevin Dynamics

In [ ]:
samples, trajectories = annealed_langevin_dynamics(
    model,
    noise_schedule,
    n_samples=CONFIG["n_generated"],
    data_dim=CONFIG["data_dim"],
    steps_per_sigma=CONFIG["steps_per_sigma"],
    device=DEVICE,
    return_trajectories=True,
)

print(f"Generated {samples.shape[0]} samples")

In [ ]:
plot_samples(data_tensor, samples.cpu())
plt.show()

## Sampling Trajectories

In [ ]:
plot_sampling_trajectory(trajectories, real_data=data_tensor, n_traces=50)
plt.show()